In [ ]:
# Statistics of the HFR returns

In [ ]:
import math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from dataclasses import asdict
from IPython.display import HTML
from functions import estimate_parameters, round_p_value, hplot
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch

In [ ]:
labels = { 
    'HFRIFWI Index': 'Composite',
    'HFRIEHI Index': 'Equity',
    'HFRIEDI Index': 'Event-Driven',
    'HFRIRVA Index': 'Relative Value',
    'HFRIMI Index':  'Macro',
}

In [ ]:
d = pd.read_csv( 'HFR Returns.csv' ).set_index( 'Date' )
d.index = pd.to_datetime( d.index )

In [ ]:
ids = list( labels.keys() ) + [ id for id in d.columns.tolist() if id not in labels ]
d = d[ids]
d.columns = [ labels[id] if id in labels else id for id in d.columns ]
d

In [ ]:
fig, axs = plt.subplots( d.shape[1], 1, figsize = (10, d.shape[1]), layout = 'constrained', dpi = 100 )
for ax, id in zip( axs, d.columns ):
    y = np.log(d[id]).diff().dropna()
    model = arch_model(y, vol='Garch', p=1, q=1, dist='skewt', mean = 'Zero', rescale=True)
    res = model.fit(disp="off")
    nlags = 5
    arch_lm_stat, arch_lm_pvalue, f_stat, f_pvalue = het_arch(y - y.mean(), nlags=nlags)
    ax.fill_between( 
        res.conditional_volatility.index,
        - res.conditional_volatility / res.scale,
        res.conditional_volatility / res.scale,
        color = 'tab:orange',
        alpha = .2,
    )
    hplot( y, ax = ax )
    ax.axis('off')
    ax.set_xlim( y.index[0], y.index[-1] )

    text = (
        f"α={res.params['alpha[1]']:.2f} "
        f"β={res.params['beta[1]']:.2f} "
        f"p={round_p_value(arch_lm_pvalue)}"
    )
    text = ax.text( 
        0, 1, 
        text,
        ha = 'left', va = 'top', 
        transform = ax.transAxes,
    )
    text.set_path_effects([
        path_effects.Stroke(linewidth=4, foreground='white', alpha = .5),
        path_effects.Normal()
    ])
    
    ax.text( 1, 1, id, ha = 'right', va = 'top', transform = ax.transAxes )
plt.show()

In [ ]:
parameters = {}
for id in d.columns:
    y = d[id].dropna()
    y = np.log( y ).diff().dropna()
    y = y[ np.isfinite(y) ]
    if len(y) < 10: 
        continue
    parameters[id] = estimate_parameters(y)
parameters = { k: asdict(v) for k, v in parameters.items() }
parameters = pd.DataFrame( parameters ).T
parameters.rename( columns = { 'denominator': '1-α²κ-2αβ-β²' }, inplace = True )

In [ ]:
tmp = parameters[[
    'skew', 'kurtosis', 
    'alpha', 'beta', 'phi', 
    'tail_index', 
    '1-α²κ-2αβ-β²', 
    #'skew_t_a', 'skew_t_b',
    'arch_lm_pvalue', 'f_pvalue', 'ljung_box_pvalue'
    ]].copy()
for column in tmp.columns: 
    if column in ['arch_lm_pvalue', 'f_pvalue', 'ljung_box_pvalue']:
        tmp[column] = tmp[column].apply( round_p_value )
    else: 
        tmp[column] = tmp[column].round(2)
display( tmp )
tmp = tmp.iloc[:5,:].T
tmp = tmp.rename( {
    'alpha': r'$\alpha$',
    'beta': r'$\beta$',
    'phi': r'$\phi = \alpha + \beta$',
    'tail_index': 'tail index',
    '1-α²κ-2αβ-β²': r'$1-\alpha^2\kappa-2\alpha\beta-\beta^2$',
    'arch_lm_pvalue': r'GARCH LM p-value',
    'f_pvalue': r'F-test p-value', 
    'ljung_box_pvalue': r'Ljung-Box p-value',
} )
tmp = tmp.to_latex()
# Remove trailing zeroes, but keep at least two digits
tmp = re.sub( r'([.][0-9][0-9][1-9]?)0+ ', r'\1 ', tmp )
tmp = re.sub( r' -([.0-9]+) ', r' $-\1$ ', tmp )
tmp = re.sub( 'llllll' , 'lrrrrr', tmp )
print( tmp )

In [ ]:
fig, axs = plt.subplots( 1, 3, figsize = (10, 2.5), layout = 'constrained', dpi = 300 )
for column, ax in zip(['arch_lm_pvalue', 'f_pvalue', 'ljung_box_pvalue'], axs):
    ax.hist( parameters[column], bins = np.linspace(0, 1, 21), density = True)
    ax.set_ylim( 0, 2.7 )
    for side in ['top', 'right', 'left']:
        ax.spines[side].set_visible(False)
    ax.set_yticks([])
    ax.set_xlabel(column)
fig.suptitle( 
    "p-values of GARCH(1,1) tests\nH₀: no (G)ARCH effect\n" 
    f"p<0.05 for {int(100 * np.mean( parameters['arch_lm_pvalue'] < .05 ))}% of time series"
)
plt.show()

In [ ]:
print( 
    f"Only {int(100 * np.mean( parameters['1-α²κ-2αβ-β²'] > 0 ))}% "
    "of time series satisfy the condition 1-α²κ-2αβ-β²>0" 
)

In [ ]:
p = parameters[['arch_lm_pvalue', 'f_pvalue', 'ljung_box_pvalue']].sort_values('arch_lm_pvalue', ascending = False)
p = p.sort_values('arch_lm_pvalue', ascending = False)
display(p)